# stats for ds

## intro
no notes

## descriptive v.s. inferential stats
- descriptive stats - describe and summarize data
- inferential stats - make inferences about a population based on a sample

## fundamental terms in distributions
- probability mass function is used for discrete data
- probability density function is used for continuous data

# binomial dist
binomial dist is a discrete dist that models the number of successes in a fixed number of trials 
- e.g. probability of $X$ heads in 10 coin flips
- e.g. probabilities of 1, 5, 10 or 20 defective parts in a batch of 100

$ P(X = x) = \binom{n}{x} p^x(1-p)^{n-x} $

$ \binom{n}{x} = \frac{n!}{x!(n-x)!} $

#### bernoulli distribution
probability of success v.s. failure for a single trial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import scipy.stats as stats
from scipy.stats import binom

#### example case

80% of all the visitors to lavista museum end up bying souvenirs from the shop at the museum. on the coming sunday, if a random sample of 10 visitors is selected:
1. find the probability that every visitor will end up buying from the shop
2. frind the prob. that a max of 7 visitors iwll buy souvenirs from the shop

do we satisfy the following assumptions?
- there are only 2 possible outcomes (success or failure) for each trial - a visitor will buy souvenirs from the shop or not (yes or no)
- number of trials (n) is fixed - there are 10 visitors in the sample
- each trial is independent of the other trials - it is reasonable to assume that the buying activity of visitors are independent
- the probability of success (p) is the same for each trial - the probability of success for each visitor is 0.8

In [ ]:
n = 10
p = 0.8
k = np.arange(0, 11) # get list of possible number of visitors from 0 to 10
print('k:', k)

In [ ]:
# use probability mass function (pmf) to calculate the probability of getting k successes
binomial = binom.pmf(k=k, n=n, p=p)

# visualization
bar1 = plt.bar(k, binomial)
plt.title(f'binomial: n={n}, p={p}', fontsize = 15)
plt.xlabel('number of successes')
plt.ylabel('probability of successes')

# color bars for 0-7 successes (sum these to get cdf)
for i in range(0, 8):
    bar1[i].set_color('r')
plt.show()

In [ ]:
def percentile_string(num):
    return format(num * 100, '.2f') + '%'

answer1 = binomial[10]
# answer2 = binomial[:8].sum()
answer2 = binom.cdf(k=7, n=n, p=p) # cumulative distribution function

print(f'Answer 1 - the probability that everyone buys something = {percentile_string(answer1)}')
print(f'Answer 2 - the probability that a max of 7 people will buy something = {percentile_string(answer2)}')

# Uniform Distribution

In [ ]:
from scipy.stats import uniform
df_debugging = pd.read_csv(f'test_data/debugging.csv')
df_debugging.head()

In [ ]:
# here se see that the data is normally distributed between 1 & 5. these are the only numbers that we need in order to create a normal distribution. see examples below.
sns.displot(df_debugging['Time Taken to fix the bug'], kde=True)
plt.show()

In [ ]:
x = np.linspace(1, 5, 50)
probs = uniform.pdf(x, loc=1, scale=4)

In [ ]:
x1 = np.linspace(1, 3, 25)
plt.plot(x, probs)
plt.fill_between(x, probs)
plt.fill_between(x1, uniform.pdf(x=x1, loc=1, scale=4), color='r')
plt.xlabel('time required for bug fixing')
plt.ylabel('probability')
plt.title('continuous uniform distribution X ~ U(1, 5)')
plt.show()

In [ ]:
# a distribution of U(1, 5) starts at 1 and has a range of (5-1) = 4

# cdf = cumulative probablity distribution function
p_less_than_x = uniform.cdf(3, loc=1, scale=4)

# ppf = percent point function
#     = the point at which the cumulative distribution function is equal to the given probability
x_given_p = uniform.ppf(0.25, loc=1, scale=4)

# pdf = probability density function
#     = the probability that the variable takes on the value x
pdf = uniform.pdf(3, loc=1, scale=4)

print('p_less_than_x (3): ', p_less_than_x)
print('x_given_p:         ', x_given_p)
print('pdf:               ', pdf)

# Normal Distribution

In [ ]:
def z_score(x, mean, std):
    # this is a normalized measure of how many standard deviations a value is from the mean
    return (x - mean) / std

def x_value(z, mean, std):
    # this is the reverse of z_score
    return z * std + mean

percentage_of_data_within_n_std = {
    1: 0.68, # 68% of the data is within 1 standard deviation
    2: 0.95, # 95% of the data is within 2 standard deviations
    3: 0.997 # 99.7% of the data is within 3 standard deviations
}

# z-score comparison example
#   -  which class is the student doing the best in?
df = pd.DataFrame()
df['class'] = ['physics', 'history', 'computer science']
df['mean'] = [47.5, 77, 33]
df['std'] = [12.3, 8.2, 7.3]
df['score'] = [56.88, 77.1, 35.55]
df['z'] = z_score(df['score'], df['mean'], df['std'])
df

In [ ]:
from scipy.stats import norm
df_sat_score = pd.read_csv('test_data/sat_score.csv')
print('mean: ', df_sat_score['score'].mean().round(2))
print('std:  ', df_sat_score['score'].std().round(2))
df_sat_score.head()

In [ ]:
# calculate the pdf of SAT scores using norm.pdf()
df_density = pd.DataFrame()
df_density['x'] = np.linspace(
    df_sat_score['score'].min() - 0.01, 
    df_sat_score['score'].max() + 0.01, 
    100
)
# for each x in X, get the probability of x given the mean and std of the SAT scores
df_density['pdf'] = norm.pdf(
    df_density['x'], 
    df_sat_score['score'].mean(), 
    df_sat_score['score'].std()
)

# plot the density
# here we see that the data very closely follows a normal distribution
fig, ax = plt.subplots()
sns.histplot(df_sat_score['score'], kde=True, ax=ax, stat='density') # raw data in blue
ax.plot(df_density['x'], df_density['pdf'], color='red')             # normal distribution in red
plt.title('SAT scores')
plt.show()

In [ ]:
mean = df_sat_score['score'].mean()
std = df_sat_score['score'].std()
x = 800
p_less_than_n = norm.cdf(x, mean, std)
print('mean: ', mean.round(2))
print('std:  ', std.round(2))
print(f'p_less_than_x ({x}): {p_less_than_n.round(2)}')

# plot to see
plt.plot(df_density['x'], df_density['pdf'])
plt.axvline(x=x, color='r', linestyle='--')
plt.fill_between(df_density['x'], df_density['pdf'], where=(df_density['x'] < x), color='r', alpha=0.3)
plt.title('Normal Distribution of SAT scores')
plt.show()

In [ ]:
# ppf = estimated score given the percentile
# - note that ppf is the reverse of cdf
p = 0.95
x_given_p = norm.ppf(p, mean, std)
print(f'x_given_p ({p}): {x_given_p.round(2)}')

# plot to see
plt.plot(df_density['x'], df_density['pdf'])
plt.axvline(x=x_given_p, color='r', linestyle='--')
plt.fill_between(df_density['x'], df_density['pdf'], where=(df_density['x'] < x_given_p), color='r', alpha=0.3)
plt.title('Normal Distribution of SAT scores')
plt.show()


# Sampling Data and Inference Foundations

**sample_std** of sample_mean = **population_std** / sqrt(n)
- as n increases, the standard deviation of the sample mean decreases
- as n increases, the sample mean becomes more accurate
- as n increases, the sample mean becomes more normally distributed (even if the original data is not normally distributed)
    - this is because the sample mean is the sum of many random variables, and the central limit theorem states that 
    - the sum of many random variables is normally distributed, regardless of the distribution of the original random variables

In [ ]:
2.1.15
2.2.7
## uniform dist theory
## continuous uniform dist - hands on
## normal dist theory
## normal dist - hands on
## z-score - hands on
## sampling and iferenece foundations
## central limit theorem theory
## central limit theorem - hands on
## estimation
## estimation - hands on
## intro to hypothesis testing
## hypothesis formation
## basic concepts of hypothesis testing
## template for hypothesis testing
## performing hypothesis test
## one tailed an two tailed tests
## confidence intervals and hypothesis testing